# Prompt Kaynaklari Wiki Sync

Prompt-writer/agent-optimizer akisi icin secilen 3 statik referans kaynagini
(bkz. AGENTS.md) ceker, ozetler ve Azure DevOps Wiki'ye
(`/Prompt-Kaynaklari/...`) yazar. Bu sayfalar daha sonra ilgili agent'in
system prompt'una statik olarak gomulmek uzere kullanilir.

Kaynaklar:
1. Anthropic - Claude Prompt Engineering (vendor)
2. The Prompt Report - Schulhoff et al., 2024 (akademik, arXiv)
3. Prompt Engineering Guide - dair-ai (topluluk, GitHub raw markdown)

robots.txt ve lisans kontrolu onceden yapildi (bkz. AGENTS.md):
- platform.claude.com: `/docs/` serbest (sadece `/api/` disallow)
- arxiv.org (ana domain): `/abs/` Allow, Crawl-delay: 15
- raw.githubusercontent.com: robots.txt yok; repo MIT lisansli


In [0]:
%pip install beautifulsoup4

In [0]:
%run "./Utils"

## Kaynak tanimlari

In [0]:
import json
import re
import html as html_module
from datetime import datetime, timezone
from urllib.parse import urlparse

from bs4 import BeautifulSoup


SOURCES = [
    {
        "name": "Anthropic Claude Prompt Engineering",
        "category": "vendor",
        "url": "https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices",
        "license": "Anthropic PBC, dahili referans amacli. Disariya yeniden yayinlanmamali.",
    },
    {
        "name": "The Prompt Report: A Systematic Survey of Prompt Engineering Techniques",
        "category": "academic",
        "url": "https://arxiv.org/abs/2406.06608",
        "license": "CC BY 4.0 (arXiv, Schulhoff et al., 2024)",
    },
]

TECHNIQUE_FILES = [
    "zeroshot",
    "fewshot",
    "cot",
    "consistency",
    "knowledge",
    "rag",
    "react",
    "tot",
    "prompt_chaining",
]

DAIR_AI_BASE = (
    "https://raw.githubusercontent.com/dair-ai/Prompt-Engineering-Guide"
    "/main/pages/techniques"
)


## robots.txt dogrulamasi (calisma zamaninda)

In [0]:
for source in SOURCES:
    allowed = check_robots_allowed(source["url"])
    print(f"{source['name']}: {'ALLOWED' if allowed else 'DISALLOWED'}")

dair_ai_sample_url = f"{DAIR_AI_BASE}/zeroshot.en.mdx"
dair_ai_allowed = check_robots_allowed(dair_ai_sample_url)
print(f"dair-ai raw markdown: {'ALLOWED' if dair_ai_allowed else 'DISALLOWED'}")


## Anthropic - Claude Prompt Engineering

Bu sayfa Next.js (React Server Components) ile render edildigi icin
gercek metin, statik HTML etiketleri icinde degil, JS payload'i icindeki
string literal'lerde tasiniyor. `extract_anthropic_content` bu payload'dan
okunabilir metin parcalarini best-effort olarak cikarir (regex tabanli).
Site yapisi degisirse bu fonksiyonun guncellenmesi gerekebilir.


In [0]:
def extract_anthropic_content(raw_html):

    quoted_strings = re.findall(r'"((?:[^"\\]|\\.){20,})"', raw_html)

    def clean(raw_string):
        try:
            text = raw_string.encode("latin-1", errors="ignore").decode("unicode_escape", errors="ignore")
        except Exception:
            text = raw_string
        text = html_module.unescape(text)
        text = re.sub(r"<[^>]*>?$", "", text)
        text = text.lstrip(">").strip()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def is_prose(text):
        if len(text) < 40:
            return False
        if "chunks/" in text or "static/" in text or "className" in text:
            return False
        if text.startswith("http") or text.startswith("/_next") or text.startswith("$"):
            return False
        words = text.split(" ")
        if len(words) < 6:
            return False
        alpha_chars = sum(c.isalpha() or c.isspace() for c in text)
        if alpha_chars / len(text) < 0.85:
            return False
        long_words = [w for w in words if len(w) > 2 and w.isalpha()]
        if len(long_words) < 5:
            return False
        return True

    cleaned_lines = [clean(s) for s in quoted_strings]
    prose_lines = [line for line in cleaned_lines if is_prose(line)]

    seen = set()
    unique_lines = []

    for line in prose_lines:
        if line not in seen:
            seen.add(line)
            unique_lines.append(line)

    return "\n\n".join(unique_lines)


## The Prompt Report (arXiv abstract)

In [0]:
def fetch_arxiv_abstract(url):

    raw_html = fetch_url_text(
        url,
        request_headers={"User-Agent": "aXet-Project-BuildAgent/1.0 (internal reference sync)"},
    )

    soup = BeautifulSoup(raw_html, "html.parser")

    title = soup.find("h1", class_="title")
    authors = soup.find("div", class_="authors")
    abstract = soup.find("blockquote", class_="abstract")
    license_tag = soup.find("div", class_="abs-license")

    title_text = title.get_text(strip=True).replace("Title:", "").strip() if title else ""
    authors_text = authors.get_text(strip=True).replace("Authors:", "").strip() if authors else ""
    abstract_text = abstract.get_text(strip=True).replace("Abstract:", "").strip() if abstract else ""
    license_text = license_tag.get_text(strip=True) if license_tag else ""

    return {
        "title": title_text,
        "authors": authors_text,
        "abstract": abstract_text,
        "license": license_text,
    }


## Prompt Engineering Guide (dair-ai)

Secilen 9 teknik dosyasi (algoritmik olanlar - APE, DSP, PAL, ART,
ActivePrompt, Reflexion - AGENTS.md kararina uygun olarak disarida
tutuldu), Ingilizce `.mdx` versiyonlari, raw.githubusercontent.com
uzerinden cekilir.


In [0]:
def fetch_dair_ai_techniques():

    sections = []

    for technique in TECHNIQUE_FILES:
        file_url = f"{DAIR_AI_BASE}/{technique}.en.mdx"
        content = fetch_url_text(file_url)
        sections.append(f"## {technique}\n\n{content.strip()}")

    return "\n\n---\n\n".join(sections)


## Agent-friendly wiki icerik olusturucu

In [0]:
def build_reference_wiki_content(source_name, source_url, license_text, fetched_at, body_text):

    metadata = {
        "source_name": source_name,
        "source_url": source_url,
        "license": license_text,
        "fetched_at": fetched_at,
        "purpose": "prompt-writer/agent-optimizer akisi icin statik referans kaynagi",
    }

    metadata_block = json.dumps(metadata, ensure_ascii=False, indent=2)

    return (
        f"# {source_name}\n\n"
        f"```json\n{metadata_block}\n```\n\n"
        f"## Icerik\n\n{body_text}\n"
    )


## Wiki'ye yazma

In [0]:
fetched_at = datetime.now(timezone.utc).isoformat()

anthropic_source = SOURCES[0]
anthropic_raw = fetch_url_text(anthropic_source["url"])
anthropic_body = extract_anthropic_content(anthropic_raw)
anthropic_content = build_reference_wiki_content(
    anthropic_source["name"],
    anthropic_source["url"],
    anthropic_source["license"],
    fetched_at,
    anthropic_body,
)

push_wiki_page("/Prompt-Kaynaklari/Anthropic-Claude-Prompt-Engineering", anthropic_content)


In [0]:
arxiv_source = SOURCES[1]
arxiv_data = fetch_arxiv_abstract(arxiv_source["url"])

arxiv_body = (
    f"**Title:** {arxiv_data['title']}\n\n"
    f"**Authors:** {arxiv_data['authors']}\n\n"
    f"**Abstract:** {arxiv_data['abstract']}\n\n"
    f"**License:** {arxiv_data['license']}"
)

arxiv_content = build_reference_wiki_content(
    arxiv_source["name"],
    arxiv_source["url"],
    arxiv_source["license"],
    fetched_at,
    arxiv_body,
)

push_wiki_page("/Prompt-Kaynaklari/The-Prompt-Report", arxiv_content)


In [0]:
dair_ai_body = fetch_dair_ai_techniques()

dair_ai_content = build_reference_wiki_content(
    "Prompt Engineering Guide (dair-ai)",
    "https://www.promptingguide.ai/",
    "MIT License (dair-ai/Prompt-Engineering-Guide)",
    fetched_at,
    dair_ai_body,
)

push_wiki_page("/Prompt-Kaynaklari/DAIR-AI-Prompt-Engineering-Guide", dair_ai_content)


## Index sayfasi

In [0]:
index_content = (
    "# Prompt Yazma Referans Kaynaklari\n\n"
    "Prompt-writer/agent-optimizer akisi icin statik olarak gomulen 3 kaynak:\n\n"
    "1. [Anthropic - Claude Prompt Engineering](/Prompt-Kaynaklari/Anthropic-Claude-Prompt-Engineering)\n"
    "2. [The Prompt Report (arXiv)](/Prompt-Kaynaklari/The-Prompt-Report)\n"
    "3. [dair-ai Prompt Engineering Guide](/Prompt-Kaynaklari/DAIR-AI-Prompt-Engineering-Guide)\n\n"
    f"Son senkronizasyon: {fetched_at}\n"
)

push_wiki_page("/Prompt-Kaynaklari", index_content)
